# Hourly Traffic Volume Forecasting on I-94 with Deep Learning
**Course:** Python for AI and Data Science (P4AIDS) — HCMUT
**Topic:** Demand Forecasting
**Dataset:** [Metro Interstate Traffic Volume — UCI ML Repository](https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume) (CC BY 4.0)

This notebook follows the four assignment requirements:

| # | Requirement | Section |
|---|---|---|
| 1 | Problem Definition | §1 |
| 2 | Literature Review | §2 |
| 3 | Algorithm Implementation (Deep Learning: preprocessing, training / fine-tuning, model explanation) | §3 – §9 |
| 4 | Evaluation — compare and analyse the implemented approaches | §10 – §12 |

**Project layout.** The code lives in the `src/metro_traffic` package so that the notebook, the command-line
script `train.py` and the Streamlit dashboard share exactly the same implementation:

```
Metro traffic/
├── data/Metro_Interstate_Traffic_Volume.csv.gz
├── src/metro_traffic/   config · data · features · baselines · metrics · models · training · explain · pipeline · artifacts
├── notebooks/P4AIDS_Metro_Traffic_DL.ipynb   (this notebook)
├── train.py              (same experiment from the command line)
├── export_results.py     (figures / tables / summary -> result/)
├── dashboard/app.py      (Streamlit dashboard)
├── outputs/              (artifacts written in §12, read by the dashboard)
└── result/               (figures, tables and summary exported in §12.1)
```

> **Running on Google Colab:** upload the whole `Metro traffic` folder to Google Drive, set `PROJECT_DIR` in the
> first code cell, choose `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (≈ 20–40 min).

## 0. Setup

In [ ]:
import os, sys, inspect

# Path of the "Metro traffic" folder on Google Drive -- change it if you put the folder elsewhere
PROJECT_DIR = "/content/drive/MyDrive/P4AIDS/Metro traffic"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:                      # running locally (e.g. Jupyter inside notebooks/)
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
print("working directory:", os.getcwd())
print("project files:", sorted(os.listdir(".")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Code, display

from metro_traffic.config import Config
from metro_traffic.data import load_raw, preprocess, data_quality_report
from metro_traffic.features import add_features, build_dataset, day_type, SEASON_LAG
from metro_traffic.pipeline import Experiment

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

def show_source(obj):
    # Display the source code of a function / class from the package
    return Code(inspect.getsource(obj), language="python")

### Configuration
All settings are in one dataclass. Set `run_tuning=False` / `run_ablation=False` for a faster run.

In [ ]:
cfg = Config(
    data_path="data/Metro_Interstate_Traffic_Volume.csv.gz",   # downloaded from UCI if missing
    out_dir="outputs",
    lookback=168, horizon=24,
    epochs=40, patience=6, batch_size=256, lr=1e-3,
    run_tuning=True, tune_top_k=2, n_trials=8, tune_epochs=15,
    run_ablation=True,
)
exp = Experiment(cfg)
cfg.to_dict()

## 1. Problem Definition

**Context.** The dataset records hourly **westbound traffic volume on Interstate 94** between Minneapolis and
St Paul (MN DoT ATR station 301) from October 2012 to September 2018, together with hourly weather
(OpenWeatherMap) and public holidays.

**Problem — multi-step demand forecasting.** At a forecast origin $t$, using the last **168 hours** of traffic,
weather and calendar data and the **calendar of the next 24 hours** (known in advance), predict the traffic
volume $\hat y_{t+1},\dots,\hat y_{t+24}$ (vehicles per hour):

$$\hat{\mathbf y}_{t+1:t+24} = f_\theta\big(\mathbf X_{t-167:t},\; \mathbf C_{t+1:t+24}\big)$$

**Why it matters.** A day-ahead forecast supports signal timing, scheduling of road works, congestion warnings
and staffing of incident-response teams — decisions for *tomorrow* made with data available *today*.

**Design decisions (based on the EDA report):**
- **No leakage:** weather in the file is *observed*, not forecast, so weather is used only for **past** hours;
  only calendar features are used for the 24 future hours.
- **Chronological split** (no shuffling): train ≤ 2016, validation = 2017, test = Jan–Sep 2018. Model selection
  uses validation only; the test set is touched once at the end.
- **Metrics:** MAE, RMSE, sMAPE, $R^2$ and **rMAE** = model MAE / seasonal-naive MAE (same hour last week).
  rMAE < 1 means the model beats the naive forecast.

## 2. Literature Review

**2.1 Classical statistical methods.** ARIMA/SARIMA (Box & Jenkins) and exponential smoothing (ETS,
Holt-Winters) are the traditional tools for seasonal series. They are interpretable and data-efficient, but assume
linear dynamics, struggle with several nested seasonalities (daily + weekly + yearly), with many exogenous
variables and with series that have gaps. The **seasonal-naive** forecast (same hour last week) is a very strong
baseline for traffic data.

**2.2 Classical machine learning.** Linear/Ridge regression, Random Forests and gradient boosting (XGBoost,
LightGBM) on lag, rolling-window, Fourier and calendar features. Very effective on tabular data but dependent
on manual feature engineering.

**2.3 Deep learning for time series.**
- **Recurrent networks — LSTM** (Hochreiter & Schmidhuber, 1997) and **GRU** (Cho et al., 2014) learn long-range
  dependencies through gating. Ma et al. (2015, *Transportation Research Part C*) were among the first to apply
  LSTMs to traffic speed forecasting and outperformed ARIMA and SVR.
- **CNN / TCN** (Bai, Kolter & Koltun, 2018): dilated causal convolutions give a large receptive field, train in
  parallel and often match or beat LSTMs.
- **Transformers** (Vaswani et al., 2017) and long-sequence variants — Informer (Zhou et al., 2021), Autoformer
  (Wu et al., 2021) and **PatchTST** (Nie et al., 2023), which turns patches of the series into tokens, reducing
  cost and improving accuracy.
- **MLP-based models:** N-BEATS (Oreshkin et al., 2020); DLinear (Zeng et al., 2023) showed that simple linear
  models can rival Transformers → **strong simple baselines are mandatory** in any comparison.
- **Temporal Fusion Transformer** (Lim et al., 2021) separates past inputs from *known future inputs* and adds
  interpretability — the idea reused in our forecast head, which receives the future calendar.
- **Spatio-temporal graph networks** (DCRNN — Li et al., 2018; STGCN — Yu et al., 2018) target multi-sensor road
  networks; not applicable here because the dataset has a single station.

**2.4 Model explanation.** Permutation feature importance (Breiman, 2001) and Integrated Gradients
(Sundararajan et al., 2017) are standard model-agnostic and gradient-based ways to find which variables and which
time lags a network relies on.

**2.5 Our approach.** Following this review we (i) build seasonal-naive, daily-naive and Ridge baselines;
(ii) implement and compare five deep architectures — **MLP, LSTM, GRU, TCN and PatchTransformer** — inside one
pipeline with a shared head that sees the future calendar; (iii) tune hyperparameters and run an ablation
without weather; (iv) explain the best model with permutation importance and Integrated Gradients.

## 3. Data loading and quality check
The EDA report found 7,629 duplicated timestamps (several weather labels merged onto one hour), a 99.9 % empty
`holiday` column, a 307-day outage (2014-08 → 2015-06) and a few sensor errors (`temp = 0 K`,
`rain_1h = 9831 mm`). We verify these findings first.

In [ ]:
raw = load_raw(cfg.data_path)
print(raw.shape)
display(raw.head())
pd.Series(data_quality_report(raw), name="value").to_frame()

**Key finding:** the `holiday` label appears **only on the 00:00 row** of each holiday. Used as is, the other
23 hours of a holiday would look like a normal working day, so the label is spread to the whole day during
preprocessing. Duplicated timestamps always carry the same traffic value — only the weather label differs.

In [ ]:
tmp = raw.drop_duplicates("date_time").copy()
tmp["hour"] = tmp.date_time.dt.hour; tmp["dow"] = tmp.date_time.dt.dayofweek
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(tmp.traffic_volume, bins=60, ax=ax[0]); ax[0].set_title("traffic_volume distribution (night vs day: bimodal)")
piv = tmp.pivot_table(index="dow", columns="hour", values="traffic_volume", aggfunc="mean")
sns.heatmap(piv, cmap="viridis", ax=ax[1]); ax[1].set_title("Mean traffic: weekday × hour")
ax[1].set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], rotation=0)
order = tmp.groupby("weather_main").traffic_volume.median().sort_values().index
sns.boxplot(data=tmp, x="weather_main", y="traffic_volume", order=order, ax=ax[2])
ax[2].tick_params(axis="x", rotation=60); ax[2].set_title("Traffic by weather type")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(17, 3))
s = tmp.set_index("date_time").traffic_volume.resample("D").mean()
ax.plot(s.index, s.values, lw=0.6); ax.set_title("Daily mean traffic — note the gap 2014-08 → 2015-06"); plt.show()

**Observations.** Strong **daily** seasonality (morning peak ≈ 7 h and afternoon peak ≈ 16–17 h on weekdays)
and **weekly** seasonality (lower weekend traffic without a morning peak) → a 168-hour (one-week) lookback is
a natural choice. Weather matters, but much less than the calendar.

## 4. Preprocessing
A deterministic pipeline (`metro_traffic.data.preprocess`):

1. drop the 17 exact duplicate rows;
2. **merge duplicated timestamps** — weather labels of the same hour become **multi-hot** columns (as recommended
   by the EDA) instead of being discarded; rare labels are grouped (`Smoke`→`Haze`, `Squall`→`Thunderstorm`);
3. **sensor errors** (`temp = 0 K`, `rain_1h > 100 mm`) → NaN, then interpolated;
4. **reindex to a complete hourly grid** — gaps ≤ `max_gap` (3 h) are time-interpolated and flagged
   `is_imputed`; longer gaps (e.g. the 307-day outage) stay NaN and are marked `valid = False`, so no training
   window crosses them;
5. **spread the holiday label** from the 00:00 row to the whole day.

In [ ]:
show_source(preprocess)

In [ ]:
exp.load_data()          # runs preprocess() + build_dataset() and keeps everything inside `exp`
hourly, ds = exp.hourly, exp.ds
print(f"hourly grid: {len(hourly):,} | valid: {int(hourly.valid.sum()):,} | "
      f"interpolated: {int(hourly.is_imputed.sum()):,} | inside long gaps: {int((~hourly.valid).sum()):,}")
print("weather multi-hot columns:", exp.wx_cols)
hourly.head()

## 5. Feature Engineering

| Group | Features | Reason |
|---|---|---|
| Target history | `traffic_scaled` | main signal, used as lagged input inside the 168-h window |
| Weather (past only) | `temp_c` (°C, standardised), `rain_log` = log1p(rain), `snow_log`, `clouds` (/100), `wx_*` multi-hot | rain and snow are extremely skewed (skew 219 and 48) → log1p |
| Calendar — cyclical | `hour_sin/cos`, `dow_sin/cos`, `doy_sin/cos` | hour 23 and hour 0 are neighbours; the model sees the cycle |
| Calendar — binary | `is_weekend`, `is_holiday` (whole day), `is_offday` = weekend ∨ holiday | holidays behave like weekends |
| Data quality | `is_imputed` | tells the model a value was interpolated |

**Known-future features** (given to the forecast head for the next 24 hours): `hour_sin/cos, dow_sin/cos,
doy_sin/cos, is_weekend, is_holiday, is_offday`.

Scalers are **fitted on training rows only** to avoid leaking validation/test statistics.

In [ ]:
show_source(add_features)

In [ ]:
print(f"{len(ds.past_cols)} past features:", ds.past_cols)
print(f"{len(ds.fut_cols)} future features:", ds.fut_cols)
print(f"target scaler (train only): mean={ds.y_mean:.1f}, std={ds.y_std:.1f}")

g = ds.feat[ds.feat.valid].copy()
g["hour"] = g.index.hour
g["day_type"] = day_type(g.is_holiday, g.is_weekend)
plt.figure(figsize=(9, 3.5))
sns.lineplot(data=g, x="hour", y="traffic", hue="day_type", errorbar=None)
plt.title("Hourly profile: holidays look like weekends, not weekdays"); plt.show()

### 5.1 Sliding windows and chronological split
One sample = forecast origin $t$: inputs are rows $[t-167, t]$, targets are $[t+1, t+24]$. A window is kept only
if none of its 192 hours lies inside a long gap. The split of a window is decided by its **target range**
(inputs may reach back into the previous split — that is genuine past data, not leakage).
To save memory we only store origin indices; windows are sliced on the GPU when a batch is built.

In [ ]:
display(ds.split_summary())
show_source(build_dataset)

## 6. Baselines
Every deep model must beat simple baselines (lesson from DLinear, §2):
- **Daily-naive:** $\hat y_{t+h} = y_{t+h-24}$ (same hour yesterday)
- **Seasonal-naive (weekly):** $\hat y_{t+h} = y_{t+h-168}$ (same hour last week) — denominator of rMAE
- **Ridge regression** on the 168 traffic lags + current weather + calendar of the next 24 hours (multi-output
  linear model, `alpha` chosen on validation).

In [ ]:
exp.run_baselines()

## 7. Deep Learning models

### 7.1 Shared architecture: *past encoder + head with the future calendar*
All networks share the same structure and differ only in the **encoder**, which makes the comparison fair:

```
X_past (B,168,F) ──► Encoder ──► embedding (B,D) ─┐
                                                  ├─ concat ─► MLP head ─► + w_h · y(t+h-168) ─► ŷ (B,24)
X_future_calendar (B,24,C) ──► flatten ───────────┘
```
The head also adds a **learned weekly-seasonal skip** $w_h\,y_{t+h-168}$: the network only has to learn the
*deviation* from the seasonal-naive forecast, which speeds up and stabilises training.

| Encoder | Idea |
|---|---|
| **MLP** | flatten the whole window → two dense layers |
| **LSTM** | 2-layer LSTM, last hidden state |
| **GRU** | like LSTM with GRU cells (fewer parameters) |
| **TCN** | residual blocks of dilated causal Conv1d (dilation 1 → 64), receptive field 509 h (7 blocks × 2 convs, kernel 3) |
| **PatchTransformer** | 168 h cut into 7 patches × 24 h (PatchTST-style), positional embedding, Transformer encoder |

In [ ]:
import torch
from metro_traffic import models, training
from metro_traffic.models import build_model, n_params, DEFAULT_HP

device, batcher = exp._torch()
print("torch", torch.__version__, "| device:", device)
for kind, hp in DEFAULT_HP.items():
    m = build_model(kind, batcher.n_past, batcher.n_fut, ds.lookback, ds.horizon, **hp)
    print(f"{kind:17s} {n_params(m):>10,d} parameters   {hp}")

In [ ]:
show_source(models)

### 7.2 Training procedure
- **Loss:** Huber (Smooth L1) on the standardised target — more robust to outliers than MSE.
- **Optimiser:** AdamW (decoupled weight decay) + **ReduceLROnPlateau** (halve the LR when validation MAE stalls).
- **Regularisation:** dropout, weight decay, **gradient clipping** (norm 1.0), **early stopping** that restores the
  weights with the best validation MAE.
- **Mixed precision (AMP)** on the GPU for speed.

In [ ]:
show_source(training.train_model)

### 7.3 Training the five architectures with default hyperparameters

In [ ]:
exp.train_default_models()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for k, h in exp.histories.items():
    ax[0].plot(h.epoch, h.train_loss, label=k); ax[1].plot(h.epoch, h.val_mae, label=k)
ax[0].set_title("Training loss (Huber, scaled)"); ax[1].set_title("Validation MAE (vehicles/hour)")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 8. Hyperparameter fine-tuning
### 8.1 Random search on the validation set
The `tune_top_k` architectures with the lowest validation MAE in §7.3 are tuned by random search
(`n_trials` trials, at most `tune_epochs` epochs each) over architecture-specific hyperparameters plus learning
rate, weight decay, batch size and head size. The best configuration found is then retrained with the full epoch
budget. **Only the validation set is used** — the test set stays untouched.

In [ ]:
from metro_traffic.models import SEARCH_SPACES, COMMON_SPACE
print("architecture search spaces:", SEARCH_SPACES)
print("common search space:", COMMON_SPACE)
exp.tune()
exp.tune_log.head(10) if len(exp.tune_log) else "tuning skipped"

### 8.2 Final model selection
The final model is the one with the **lowest validation MAE** among all default and tuned runs.

In [ ]:
best = exp.select_best()
best

### 8.3 Ablation: do weather features help?
The best configuration is retrained **without any weather feature** (only traffic history + calendar). If the
error barely changes, past weather contributes little; if it increases clearly, weather is useful.

In [ ]:
exp.ablation()

## 9. Model explanation (best model)
### 9.1 Permutation feature importance (by feature group)
Each feature group is shuffled across test windows (breaking its link with the target) and we measure **how much
the MAE increases**. The larger the increase, the more the model relies on that group.

In [ ]:
exp.explain()
p = exp.perm_importance.sort_values("delta_MAE")
plt.figure(figsize=(9, 4.5))
plt.barh(p.group, p.delta_MAE, xerr=p["std"], color="#4C72B0")
plt.xlabel("Increase in test MAE when shuffled (vehicles/hour)")
plt.title(f"Permutation importance — {exp.best['name']} (base MAE = {p.base_MAE.iloc[0]:.0f})")
plt.tight_layout(); plt.show()
exp.perm_importance

### 9.2 Integrated Gradients — which past hours does the model look at?
Integrated Gradients (Sundararajan et al., 2017) integrates the gradient along a straight path from a baseline
(0 = the training mean after standardisation) to the real input:
$$\mathrm{IG}_i(x) = (x_i - x'_i)\int_0^1 \frac{\partial f(x' + \alpha(x-x'))}{\partial x_i}\,d\alpha$$
We average |IG| over 512 test windows, by time lag and by feature.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(17, 4.5), gridspec_kw=dict(width_ratios=[2, 1]))
lag = exp.ig_lag
ax[0].plot(lag.hours_before_origin, lag.importance); ax[0].invert_xaxis()
for d in range(24, 169, 24): ax[0].axvline(d, color="grey", ls=":", lw=0.8)
ax[0].set_xlabel("Hours before the forecast origin"); ax[0].set_title("Importance by time lag (sum of |IG| over features)")
exp.ig_features.head(15).sort_values("importance").plot.barh(x="feature", y="importance", ax=ax[1], legend=False)
ax[1].set_title("Top-15 past features by |IG|")
plt.tight_layout(); plt.show()

hm = exp.ig_heatmap
plt.figure(figsize=(15, 3.8))
sns.heatmap(hm.div(hm.max(axis=1), axis=0), cmap="magma", xticklabels=24)
plt.xlabel("Hours before origin (traffic history)"); plt.ylabel("Forecast step")
plt.title("IG of the traffic history for each forecast step (row-normalised)"); plt.tight_layout(); plt.show()

**How to read it:** peaks near lag 0 (recent momentum) and at ≈ 24 h and 168 h (same hour yesterday / last week)
mean the network learned **daily/weekly seasonality and short-term inertia** by itself, without hand-made lag
features. For far horizons (t+24) the attention usually moves from the most recent hours to the seasonal lags.

## 10. Evaluation — comparison on the test set (Jan–Sep 2018)

In [ ]:
res = exp.tracker.table()
display(res.style.format({"MAE": "{:.1f}", "RMSE": "{:.1f}", "sMAPE": "{:.2f}", "R2": "{:.4f}", "rMAE": "{:.3f}",
                          "val_MAE": "{:.1f}", "params": "{:,.0f}", "train_sec": "{:.0f}"}, na_rep="—")
                  .background_gradient(subset=["MAE", "RMSE"], cmap="RdYlGn_r"))

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
r = res.sort_values("MAE", ascending=False)
colors = ["#4C72B0" if n.startswith(("DL", "Ablation")) else "#C44E52" for n in r.index]
ax[0].barh(r.index, r.MAE, color=colors); ax[0].set_title("Test MAE (red = baseline, blue = deep learning)")
ax[1].barh(r.index, r.rMAE, color=colors); ax[1].axvline(1, color="k", ls="--"); ax[1].set_title("rMAE (vs seasonal-naive)")
plt.tight_layout(); plt.show()

### 10.1 Error by forecast horizon

In [ ]:
P, Y = exp.tracker.preds, exp.tracker.y_true
plt.figure(figsize=(12, 4.5))
for n in P:
    if n.startswith("Ablation") or n == "Daily-naive":
        continue
    plt.plot(np.arange(1, ds.horizon + 1), np.abs(P[n] - Y).mean(0), marker="o", ms=3, label=n,
             ls="-" if n.startswith("DL") else "--")
plt.xlabel("Hours ahead"); plt.ylabel("MAE"); plt.title("MAE by forecast step"); plt.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

### 10.2 Forecast examples
Day-ahead forecasts issued at 23:00 for the next 24 hours, stitched over one week, including weeks with holidays
(Memorial Day 2018-05-28, Independence Day 2018-07-04).

In [ ]:
BEST_KEY = exp.best["name"] if exp.best else "Ridge (linear, lags)"
pos = {o: i for i, o in enumerate(ds.idx_test)}

def plot_week(start, ax, models=(BEST_KEY, "Seasonal-naive (weekly)")):
    ts, ys, ps = [], [], {m: [] for m in models}
    for d in range(7):
        o = ds.times.get_loc(pd.Timestamp(start) + pd.Timedelta(days=d) - pd.Timedelta(hours=1))
        if o not in pos: continue
        i = pos[o]; ts += list(ds.times[o + ds.fut_off]); ys += list(Y[i])
        for m in models: ps[m] += list(P[m][i])
    ax.plot(ts, ys, color="k", lw=1.6, label="Actual")
    for m, c in zip(models, ["#4C72B0", "#C44E52"]):
        ax.plot(ts, ps[m], lw=1.1, color=c, ls="-" if m == models[0] else "--", label=m)
    ax.set_title(f"Week starting {pd.Timestamp(start).date()}"); ax.legend(loc="upper right", fontsize=8)

fig, ax = plt.subplots(3, 1, figsize=(16, 11))
for a, s in zip(ax, ["2018-03-05", "2018-05-26", "2018-07-01"]): plot_week(s, a)
plt.tight_layout(); plt.show()

### 10.3 Error analysis: hour of day, day type and residuals

In [ ]:
tpos = ds.target_pos(ds.idx_test).ravel()
err = pd.DataFrame({"time": ds.times[tpos], "y": Y.ravel(),
                    "day_type": day_type(ds.feat.is_holiday.values[tpos], ds.feat.is_weekend.values[tpos])})
err["hour"] = err.time.dt.hour
cmp = [BEST_KEY, "Ridge (linear, lags)", "Seasonal-naive (weekly)"]
cmp = list(dict.fromkeys(cmp))
for n in cmp: err[n] = np.abs(P[n].ravel() - err.y)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
err.groupby("hour")[cmp].mean().plot(ax=ax[0]); ax[0].set_title("MAE by hour of day")
err.groupby("day_type")[cmp].mean().plot.bar(ax=ax[1], rot=0); ax[1].set_title("MAE by day type")
resid = P[BEST_KEY].ravel() - err.y
sns.histplot(resid, bins=80, ax=ax[2]); ax[2].set_title(f"Residuals of {BEST_KEY} (mean={resid.mean():.1f}, std={resid.std():.0f})")
plt.tight_layout(); plt.show()
display(err.groupby("day_type")[cmp].mean().round(1))

print("Days with the largest errors (often storms, accidents, events or mid-week holidays):")
err.assign(date=err.time.dt.date).groupby("date").agg(MAE=(BEST_KEY, "mean"), mean_traffic=("y", "mean")) \
   .sort_values("MAE", ascending=False).head(8)

## 11. Automatic summary of the results

In [ ]:
R = exp.tracker.results
sn, rg, bm = R["Seasonal-naive (weekly)"], R["Ridge (linear, lags)"], R[BEST_KEY]
dl = res[res.index.str.startswith("DL")]
print(f"• Best model (selected on validation): {BEST_KEY}")
print(f"    test MAE = {bm['MAE']:.1f} veh/h, RMSE = {bm['RMSE']:.1f}, sMAPE = {bm['sMAPE']:.2f}%, R² = {bm['R2']:.4f}")
print(f"• vs seasonal-naive (MAE {sn['MAE']:.1f}): {100*(1-bm['MAE']/sn['MAE']):.1f}% lower MAE (rMAE = {bm['rMAE']:.3f})")
print(f"• vs Ridge (MAE {rg['MAE']:.1f}): {100*(1-bm['MAE']/rg['MAE']):.1f}% lower MAE")
if len(dl):
    print("• Deep models ranked by test MAE: " + " < ".join(f"{n.replace('DL-', '')} ({v:.0f})" for n, v in dl.MAE.items()))
abl = [k for k in R if k.startswith("Ablation")]
if abl:
    a = R[abl[0]]
    print(f"• Ablation without weather: MAE {a['MAE']:.1f} vs {bm['MAE']:.1f} -> weather "
          + ("reduces" if a['MAE'] > bm['MAE'] else "does NOT reduce") + f" the error by {abs(a['MAE'] - bm['MAE']):.1f} veh/h")
if exp.perm_importance is not None:
    print("• Top-3 feature groups (permutation): " + ", ".join(exp.perm_importance.group.head(3)))

## 12. Save artifacts for the dashboard
`exp.save()` writes metrics, test predictions, training histories, the tuning log, explanations, the cleaned
hourly data and the best model weights to `outputs/`. On Colab this folder is on your Google Drive.

To open the dashboard on your computer: download the `outputs/` folder into the local `Metro traffic` folder, then
```bash
pip install -r requirements.txt
streamlit run dashboard/app.py
```

In [ ]:
exp.save()
sorted(os.listdir(cfg.out_dir))

### 12.1 Export figures, tables and a summary to `result/`
`export_results` turns the saved artifacts into high-resolution PNG figures, CSV tables and a markdown summary
(`result/summary.md`) that can be reused directly in the written report.

In [ ]:
from metro_traffic.artifacts import load_artifacts
from metro_traffic.report import export_results
files = export_results(load_artifacts(cfg.out_dir), "result")
print(len(files), "files written to result/")
sorted(os.listdir("result/figures"))

## 13. Discussion and conclusion

*(Read the numbers printed in §11 and adjust the statements below to match your run.)*

**Comparison of approaches.**
- The seasonal-naive forecast is a strong baseline because weekly traffic patterns are very stable. Ridge on 168
  lags already exploits seasonality and short-term inertia and usually improves clearly on the naive forecasts.
- The deep models learn **non-linear** interactions between the future calendar (hour/day/holiday), recent
  momentum and weather, which reduces the error — especially in **rush hours** and on **holidays**, where the
  seasonal-naive forecast fails (last week was a working day, today is a holiday).
- Among the deep models, the recurrent (LSTM/GRU) and convolutional (TCN) encoders are usually close;
  PatchTransformer is light and fast because it only processes 7 tokens; the MLP is simple but overfits more
  easily. Differences between deep models are much smaller than the gap between deep models and naive
  baselines — consistent with Zeng et al. (2023): complex architectures do not automatically win.
- The error grows with the horizon (t+1 is easier than t+24), as expected for multi-step forecasting.

**Model explanation.** Permutation importance and Integrated Gradients show that the model relies mostly on
**traffic history** and the **future calendar**; past weather has a small contribution (consistent with the
ablation). IG peaks at the most recent hours and at seasonal lags (≈ 24 h, ≈ 168 h), so the network discovered
daily and weekly seasonality by itself.

**Limitations.**
1. Future weather is not used (to avoid leakage), so an upcoming snowstorm cannot be anticipated; in production,
   weather *forecasts* could be fed to the head.
2. Unusual events (accidents, road works, sports events) are not in the data → the worst days in §10.3.
3. Single station, a 307-day outage, and holidays inferred only from labels in the data.
4. Results come from one seed; multiple seeds or rolling-origin cross-validation would give confidence intervals.

**Future work.** Quantile loss for prediction intervals; an ensemble of the best deep models; LightGBM as a strong
classical-ML baseline; full TFT/PatchTST implementations; event features (sports calendar, school terms).

## References
1. Hogue, J. (2019). *Metro Interstate Traffic Volume* [Dataset]. UCI Machine Learning Repository.
2. Box, G. E. P., & Jenkins, G. M. (1970). *Time Series Analysis: Forecasting and Control*.
3. Hochreiter, S., & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation*, 9(8).
4. Cho, K. et al. (2014). Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation. *EMNLP*.
5. Ma, X. et al. (2015). Long short-term memory neural network for traffic speed prediction using remote microwave sensor data. *Transportation Research Part C*, 54.
6. Bai, S., Kolter, J. Z., & Koltun, V. (2018). An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling. *arXiv:1803.01271*.
7. Vaswani, A. et al. (2017). Attention Is All You Need. *NeurIPS*.
8. Li, Y. et al. (2018). Diffusion Convolutional Recurrent Neural Network: Data-Driven Traffic Forecasting. *ICLR*.
9. Yu, B., Yin, H., & Zhu, Z. (2018). Spatio-Temporal Graph Convolutional Networks. *IJCAI*.
10. Oreshkin, B. et al. (2020). N-BEATS: Neural basis expansion analysis for interpretable time series forecasting. *ICLR*.
11. Zhou, H. et al. (2021). Informer. *AAAI*. — Wu, H. et al. (2021). Autoformer. *NeurIPS*.
12. Lim, B. et al. (2021). Temporal Fusion Transformers for interpretable multi-horizon time series forecasting. *International Journal of Forecasting*, 37(4).
13. Nie, Y. et al. (2023). A Time Series is Worth 64 Words: Long-term Forecasting with Transformers (PatchTST). *ICLR*.
14. Zeng, A. et al. (2023). Are Transformers Effective for Time Series Forecasting? *AAAI*.
15. Breiman, L. (2001). Random Forests. *Machine Learning*, 45. — Sundararajan, M. et al. (2017). Axiomatic Attribution for Deep Networks. *ICML*.